# Random Forest на GPU. ранние пропуски в лагах
Пилот на h2 и h6. В Kaggle включите GPU, подключите train_solo_track.parquet и feature_experiments_by_horizon.csv от исходного эксперимента. При нескольких копиях укажите пути в первой ячейке. Результаты сохраняются в /kaggle/working/rf_warmup.

Полный train повторяет исходный фильтр доступности profile28. Во втором варианте удаляется только начальная часть обучающих строк, пока недоступна полная группа смещений первого недельного лага. История для расчёта признаков остаётся полной. Остальные пропуски заполняются медианой соответствующего признака по train данной модели.

На шести недельных окнах до test используем сохранённые результаты полного train и обучаем вариант с отсечением. Для независимой оценки заново обучаем оба варианта на истории до начала последних 14 дней. Признаки, параметры леса, даты и маршруты оценки внутри каждой пары совпадают.


In [ ]:
from pathlib import Path
import gc
import json
import re
import tempfile
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.impute import SimpleImputer
import cupy as cp
from cuml.ensemble import RandomForestRegressor

TRAIN_PATH = None
COMPARISON_PATH = None
HORIZONS = (2, 6)
EXPERIMENT_NAMES = ('baseline', 'thin_offsets', 'compact_with_additions')
BASE_PARAMS_BY_HORIZON = {2: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 24, 'max_features': 0.3406585285177396, 'max_samples': 0.9330880728874675}, 6: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 23, 'max_features': 0.5045012539746527, 'max_samples': 0.6831809216468459}}
SEED = 42
N_STREAMS = 4
PREDICTION_BATCH = 32768
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TRAIN_MINUTES = (570, 600, 630, 660, 690)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
RESIDUAL = False
if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError('Включите GPU в настройках Kaggle.')
cp.zeros(1, dtype=cp.float32)
cp.cuda.get_current_stream().synchronize()

def resolve_input(path, name):
    if path is None:
        matches = sorted(Path('/kaggle/input').rglob(name))
        if len(matches) != 1:
            raise ValueError(f'Укажите путь к {name} в первой ячейке. Найдено: {matches}')
        path = matches[0]
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

TRAIN_PATH = resolve_input(TRAIN_PATH, 'train_solo_track.parquet')
COMPARISON_PATH = resolve_input(COMPARISON_PATH, 'feature_experiments_by_horizon.csv')
REPORT = Path('/kaggle/working/rf_warmup')
REPORT.mkdir(parents=True, exist_ok=True)
cache = tempfile.TemporaryDirectory(prefix='matrices_', dir=REPORT)
OUT = Path(cache.name)

## Выбор признаков по шести окнам до test


In [ ]:
comparison = pd.read_csv(COMPARISON_PATH)
score = 'метрика конкурса'
keys = ['horizon', 'experiment', 'fold', 'выборка']
comparison = comparison.loc[comparison.variant.eq('direct') & comparison.horizon.isin(HORIZONS)
                            & comparison.experiment.isin(EXPERIMENT_NAMES)].copy()
expected = pd.MultiIndex.from_product([HORIZONS, EXPERIMENT_NAMES, range(1, 7), ['train', 'tuning']], names=keys)
actual = pd.MultiIndex.from_frame(comparison[keys])
if actual.has_duplicates or not expected.difference(actual).empty or not actual.difference(expected).empty:
    raise ValueError('Нужны строки train/tuning всех шести фолдов и трёх наборов для h2 и h6.')
if not np.isfinite(comparison[score]).all():
    raise ValueError('Неконечная метрика исходного эксперимента.')
ranking = (comparison.loc[comparison['выборка'].eq('tuning')].groupby(['horizon', 'experiment'])[score]
           .mean().reset_index().sort_values(['horizon', score, 'experiment']))
winners = ranking.drop_duplicates('horizon').set_index('horizon').experiment.to_dict()
display(ranking)
print('Наборы для обеих моделей:', winners)
ranking.to_csv(REPORT / 'feature_selection.csv', index=False)

In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
end = times[-1]
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
selection_end = pd.Timestamp('2025-10-11')
if len(dates) < 14 or dates[-14].normalize() < selection_end or times[-1] < selection_end:
    raise ValueError('Недостаточно истории для отдельного test после подбора')
holdout_start = dates[-14]
selection_dates = dates[dates + 8 * STEP < selection_end]
if len(selection_dates) <= 49:
    raise ValueError('До 20 сентября недостаточно истории для train и трёх недель подбора')
folds = []
for start in range(len(selection_dates) - 42, len(selection_dates), 7):
    folds.append(dict(train=selection_dates[:start-7], val=selection_dates[start-7:start],
                      tuning=selection_dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))

## Признаки исходного эксперимента


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    if lag_table is None:
        return correlation_block(frame,h,None,residual_signal,'residual_')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

In [ ]:
def stable_ratio(numerator, denominator):
    return numerator / np.maximum(np.abs(denominator), 1.0)


def available_statistics(window):
    valid = np.isfinite(window)
    count = valid.sum(axis=0)
    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count,
                     out=np.full(window.shape[1:], np.nan), where=count > 0)
    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count,
                         out=np.full_like(mean, np.nan), where=count > 0)
    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)
    return mean, np.sqrt(variance), spread, count


def add_experimental_features(frame, h, origins):
    additions = {}
    additions['x_time_cutoff_minutes'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))
    additions['x_profile_profile28'] = frame.profile28.to_numpy()
    additions['x_level_level24'] = level24[origins].ravel()
    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])
    current = target[origins]
    additions['x_anomaly_target_difference'] = (current - typical).ravel()
    additions['x_anomaly_target_ratio'] = stable_ratio(current, typical).ravel()
    for name in (TARGET, *STATUSES[:3]):
        short, long = means[name, 4][origins], means[name, 12][origins]
        additions[f'x_trend_{name}_difference'] = (short - long).ravel()
        additions[f'x_trend_{name}_ratio'] = stable_ratio(short, long).ravel()
        for steps in (4, 12):
            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])
            mean, std, spread, count = available_statistics(window)
            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]
            additions[f'x_volatility_{name}_std_{steps}'] = std.ravel()
            additions[f'x_volatility_{name}_range_{steps}'] = spread.ravel()
            additions[f'x_volatility_{name}_change_{steps}'] = change.ravel()
    for label, days in [('daily7', range(1,8)), ('weekly4', (7,14,21,28))]:
        window = np.stack([past(target, origins + h - day * 48) for day in days])
        mean, std, spread, count = available_statistics(window)
        additions[f'x_dispersion_{label}_std'] = std.ravel()
        additions[f'x_dispersion_{label}_relative_std'] = stable_ratio(std, mean).ravel()
        additions[f'x_dispersion_{label}_count'] = count.ravel()
    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)


def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    frame = add_experimental_features(frame, h, origins)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    minutes = times.hour * 60 + times.minute
    origins = np.flatnonzero(np.isin(minutes, TRAIN_MINUTES) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    if path.exists():
        print(f'h={h}: общие признаки загружены из кэша')
        return
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 35):
            batch = origins[start:start+35]
            frame = horizon_dataset(h, batch)
            frame['y'] = target[batch+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary,table.schema,compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    print(f'h={h}: история {len(origins)*len(routes):,}, модельных признаков {len(frame.columns) - 5}')
for h in HORIZONS:
    write_horizon(h)

In [ ]:
def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()

In [ ]:
def model_target(frame):
    baseline = frame.profile28.to_numpy() if RESIDUAL else 0
    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)

def transformed_frame(frame, columns):
    result = frame[columns].drop(columns='route_id').astype(np.float32)
    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1
    bits = {f'route_bit_{i}': ((codes >> i) & 1).astype(np.float32)
            for i in range(max(1, int(len(routes)).bit_length()))}
    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()

def cached_lags(boundary):
    path = OUT / f'lags_{pd.Timestamp(boundary).strftime("%Y%m%dT%H%M%S")}.parquet'
    if path.exists():
        return pd.read_parquet(path)
    table = fit_lags(boundary)
    temporary = path.with_suffix('.partial')
    table.to_parquet(temporary, index=False)
    temporary.replace(path)
    return table

def fit_forest(frame, matrix, params):
    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type='numpy', **params)
    model.fit(matrix, model_target(frame))
    cp.cuda.get_current_stream().synchronize()
    return model

def predict_matrix(model, frame, matrix):
    result = np.empty(len(frame), dtype=np.float64)
    for start in range(0, len(frame), PREDICTION_BATCH):
        stop = min(start + PREDICTION_BATCH, len(frame))
        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)
        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0
        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)
    if not np.isfinite(result).all():
        raise ValueError('Некорректный прогноз')
    return result

EXPERIMENTS = {
    'baseline': dict(),
    'drop_constant': dict(drop_constant=True),
    'compact_availability': dict(compact_availability=True),
    'calendar_dayofweek': dict(calendar=True),
    'thin_offsets': dict(thin=True),
    'drop_network': dict(network='drop'),
    'thin_network': dict(network='thin'),
    'add_time': dict(add=['time']),
    'add_profile': dict(add=['profile']),
    'add_level24': dict(add=['level']),
    'add_anomaly': dict(add=['anomaly']),
    'add_trends': dict(add=['trend']),
    'add_volatility': dict(add=['volatility']),
    'add_seasonal_dispersion': dict(add=['dispersion']),
    'all_additions': dict(add=['time','profile','level','anomaly','trend','volatility','dispersion']),
    'compact': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin'),
    'compact_with_additions': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin',
                                   add=['time','profile','level','anomaly','trend','volatility','dispersion'])
}


def experiment_columns(names, spec):
    result = []
    for name in names:
        if name.startswith('x_'):
            if any(name.startswith(f'x_{group}_') for group in spec.get('add', [])):
                result.append(name)
            continue
        if spec.get('drop_constant') and name == 'selected_status6_pair_count':
            continue
        if spec.get('compact_availability') and 'signal_available' in name:
            continue
        if spec.get('calendar') and name in ('weekday_sin','weekday_cos','is_weekend'):
            continue
        if spec.get('network') == 'drop' and name.startswith('network_'):
            continue
        if spec.get('network') == 'thin' and name.startswith('network_') and '_offset_' in name:
            continue
        if spec.get('thin') and not name.startswith('slot_') and re.search(r'_offset_m[134]$', name):
            continue
        result.append(name)
    if spec.get('compact_availability'):
        for name in ('status6_lag_valid', 'residual_status6_lag_valid'):
            if name in names and name not in result:
                result.append(name)
    return result

## Сравнение на одинаковых строках оценки


In [ ]:
def run_fit(h, mode, boundary, eval_dates, fold=None):
    started = time.perf_counter()
    experiment = winners[h]
    path = OUT / f'history_h{h}.parquet'
    lag_names = [offset_name('slot_days_back_7', k) for k in WINDOW_OFFSETS]
    correlations = cached_lags(boundary)
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    rows = []
    for split in ('train', 'tuning' if fold is not None else 'test'):
        filters = [('timestamp', '<', boundary)] if split == 'train' else [('cutoff', 'in', list(eval_dates))]
        frame = pd.read_parquet(path, filters=filters)
        frame = frame.loc[np.isfinite(frame.profile28)].copy()
        if split == 'train':
            before = len(frame)
            if mode == 'trimmed':
                available = frame[lag_names].notna().all(axis=1)
                first_ready = (frame.loc[available].groupby('route_id').cutoff.min())
                starts = frame.route_id.map(first_ready)
                frame = frame.loc[starts.notna() & frame.cutoff.ge(starts)].copy()
            removed = before - len(frame)
            if frame.empty or frame.timestamp.max() >= eval_dates.min().normalize():
                raise ValueError('Нарушена граница train.')
        elif frame.empty or set(frame.cutoff.unique()) != set(eval_dates) or len(frame) != len(routes) * len(eval_dates):
            raise ValueError('Неполная выборка оценки.')
        first = frame.cutoff.min()
        frame = add_correlation_features(frame, h, correlations)
        encoded = transformed_frame(frame, model_columns(frame))
        columns = experiment_columns(list(encoded.columns), EXPERIMENTS[experiment])
        subset = encoded[columns]
        del encoded
        matrix = imputer.fit_transform(subset) if split == 'train' else imputer.transform(subset)
        del subset
        labels = frame[['cutoff','route_id','timestamp','y','profile28']].copy()
        del frame
        if split == 'train':
            model = fit_forest(labels, matrix, BASE_PARAMS_BY_HORIZON[h])
        pred = predict_matrix(model, labels, matrix)
        rows.append(dict(horizon=h, experiment=experiment, mode=mode, fold=fold, removed_train=removed,
                         first_cutoff=str(first), features=len(columns), **score_row(labels, pred, split)))
        del labels, matrix, pred
        gc.collect()
    print(f'h{h}, {mode}, {time.perf_counter()-started:.0f} секунд')
    display(pd.DataFrame(rows))
    return rows

In [ ]:
saved = comparison.loc[comparison.experiment.eq(comparison.horizon.map(winners))].assign(mode='full_train', removed_train=0)
rows = saved.to_dict('records')
for h in HORIZONS:
    for number, f in enumerate(folds, 1):
        rows.extend(run_fit(h, 'trimmed', f['val'][0].normalize(), f['tuning'], number))
        pd.DataFrame(rows).to_csv(REPORT / 'fold_metrics.csv', index=False)

def paired_scores(records, split, index):
    frame = pd.DataFrame(records)
    result = frame.loc[frame['выборка'].eq(split)].pivot(index=index, columns='mode', values=score)
    result['delta_trimmed_minus_full'] = result.trimmed - result.full_train
    return result

weeks = paired_scores(rows, 'tuning', ['horizon', 'fold'])
display(weeks)
means = weeks.groupby('horizon').mean()
means['chosen_on_tuning'] = np.where(means.trimmed < means.full_train, 'trimmed', 'full_train')
display(means)
means.to_csv(REPORT / 'selection_before_test.csv')

## Оценка на test на последних 14 днях
Правило выбора зафиксировано в selection_before_test.csv. Отрицательная разница означает меньшую ошибку при отсечении строк.


In [ ]:
final_rows = []
for h in HORIZONS:
    for mode in ('full_train', 'trimmed'):
        final_rows.extend(run_fit(h, mode, holdout_start.normalize(), dates[-14:]))
        pd.DataFrame(final_rows).to_csv(REPORT / 'independent_train_test.csv', index=False)
for split in ('train', 'test'):
    print(split)
    display(paired_scores(final_rows, split, ['horizon']).join(means[['chosen_on_tuning']]))
print('Независимая оценка test получена до переобучения на всей истории. Эксперимент заканчивается сравнением моделей.')
print(REPORT)